<a href="https://colab.research.google.com/github/kb0417/french-ewe-translation-transcription/blob/main/notebooks/08_gradio_translation_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 08 — Interface de démonstration avec Gradio

Dans ce notebook, nous créons une interface simple permettant de tester le système de traduction automatique français-éwé.

L’utilisateur peut choisir la direction de traduction :

- français → éwé ;
- éwé → français.

L’interface utilise les modèles NLLB fine-tunés avec LoRA dans les notebooks précédents.

In [1]:
# On installe les bibliothèques nécessaires.
# gradio permet de créer rapidement une interface web de démonstration.
# transformers et peft permettent de recharger les modèles NLLB fine-tunés avec LoRA.

!pip install -U gradio transformers sentencepiece accelerate peft torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 126.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 9.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires starlette<1.0.0,>=0.49.1, but you have starlette 1.2.1 which is incompatible.


In [2]:
import os
import torch
import gradio as gr

from google.colab import drive

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

In [3]:
# On connecte Google Drive pour accéder aux modèles fine-tunés sauvegardés.

drive.mount('/content/drive')

project_dir = "/content/drive/MyDrive/french_ewe_project"
models_dir = os.path.join(project_dir, "models")

print("Dossier modèles :", models_dir)
print(os.listdir(models_dir))

Mounted at /content/drive
Dossier modèles : /content/drive/MyDrive/french_ewe_project/models
['best_seq2seq_fr_to_ewe.keras', 'seq2seq_lstm_fr_to_ewe_final.keras', 'best_seq2seq_ewe_to_fr.keras', 'history_seq2seq_ewe_to_fr.csv', 'seq2seq_lstm_ewe_to_fr_final.keras', 'nllb_fr_to_ewe_lora', 'nllb_fr_to_ewe_lora_final', 'nllb_ewe_to_fr_lora', 'nllb_ewe_to_fr_lora_final']


In [4]:
# Chemins des adaptateurs LoRA fine-tunés.

fr_to_ewe_lora_path = os.path.join(models_dir, "nllb_fr_to_ewe_lora_final")
ewe_to_fr_lora_path = os.path.join(models_dir, "nllb_ewe_to_fr_lora_final")

print("Modèle français → éwé :", fr_to_ewe_lora_path)
print("Modèle éwé → français :", ewe_to_fr_lora_path)

Modèle français → éwé : /content/drive/MyDrive/french_ewe_project/models/nllb_fr_to_ewe_lora_final
Modèle éwé → français : /content/drive/MyDrive/french_ewe_project/models/nllb_ewe_to_fr_lora_final


In [5]:
# On utilise le même modèle de base que pendant le fine-tuning.
# Les adaptateurs LoRA seront ensuite chargés par-dessus ce modèle.

base_model_name = "facebook/nllb-200-distilled-600M"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Appareil utilisé :", device)

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)
base_model = base_model.to(device)

print("Modèle NLLB de base chargé.")

Appareil utilisé : cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Modèle NLLB de base chargé.


In [6]:
# On charge le modèle fine-tuné français → éwé.
# PeftModel applique l'adaptateur LoRA sur le modèle NLLB de base.

model_fr_to_ewe = PeftModel.from_pretrained(
    base_model,
    fr_to_ewe_lora_path
)

model_fr_to_ewe = model_fr_to_ewe.to(device)
model_fr_to_ewe.eval()

print("Adaptateur LoRA français → éwé chargé.")

Adaptateur LoRA français → éwé chargé.


In [7]:
# On recharge un deuxième modèle de base pour l'adaptateur éwé → français.
# C'est plus clair et plus sûr pour une démonstration.

base_model_2 = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)
base_model_2 = base_model_2.to(device)

model_ewe_to_fr = PeftModel.from_pretrained(
    base_model_2,
    ewe_to_fr_lora_path
)

model_ewe_to_fr = model_ewe_to_fr.to(device)
model_ewe_to_fr.eval()

print("Adaptateur LoRA éwé → français chargé.")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Adaptateur LoRA éwé → français chargé.


In [8]:
# Codes de langue utilisés par NLLB.

FRENCH_CODE = "fra_Latn"
EWE_CODE = "ewe_Latn"

In [9]:
def translate_with_nllb(text, direction):
    # On vérifie que l'utilisateur a bien saisi une phrase.
    if text is None or text.strip() == "":
        return "Veuillez saisir une phrase à traduire."

    # Direction français → éwé
    if direction == "Français → Éwé":
        src_lang = FRENCH_CODE
        tgt_lang = EWE_CODE
        model = model_fr_to_ewe

    # Direction éwé → français
    elif direction == "Éwé → Français":
        src_lang = EWE_CODE
        tgt_lang = FRENCH_CODE
        model = model_ewe_to_fr

    else:
        return "Direction de traduction inconnue."

    # On indique la langue source au tokenizer.
    tokenizer.src_lang = src_lang

    # Tokenisation de la phrase.
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)

    # Génération de la traduction.
    with torch.no_grad():
        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
            max_length=128,
            num_beams=4
        )

    # Décodage des tokens générés en texte lisible.
    translation = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    return translation

In [10]:
print(translate_with_nllb("Je suis malade", "Français → Éwé"))
print(translate_with_nllb("Mele dɔ lém", "Éwé → Français"))

mele dɔ lém
Je suis malade


In [11]:
# On crée une interface simple avec :
# - une zone de texte en entrée ;
# - un menu de sélection pour la direction ;
# - une zone de texte en sortie.

demo = gr.Interface(
    fn=translate_with_nllb,

    inputs=[
        gr.Textbox(
            label="Texte à traduire",
            placeholder="Entrez une phrase en français ou en éwé..."
        ),
        gr.Radio(
            choices=["Français → Éwé", "Éwé → Français"],
            value="Français → Éwé",
            label="Direction de traduction"
        )
    ],

    outputs=gr.Textbox(
        label="Traduction générée"
    ),

    title="Système de traduction automatique français-éwé",
    description=(
        "Interface de démonstration utilisant NLLB fine-tuné avec LoRA "
        "pour la traduction français ↔ éwé."
    ),

    examples=[
        ["Je suis malade", "Français → Éwé"],
        ["Merci beaucoup", "Français → Éwé"],
        ["Je vais à la maison", "Français → Éwé"],
        ["Meda akpe", "Éwé → Français"],
        ["Mele dɔ lém", "Éwé → Français"],
        ["Meyina aƒe me", "Éwé → Français"]
    ]
)

In [12]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5d83d1ef9c144823c0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Conclusion

Dans ce notebook, nous avons créé une interface de démonstration avec Gradio pour notre système de traduction automatique français-éwé.

L’interface permet à l’utilisateur de saisir une phrase, de choisir la direction de traduction, puis d’obtenir une traduction générée automatiquement.

Les deux directions sont prises en charge :

- français → éwé ;
- éwé → français.

Les modèles utilisés sont les versions NLLB fine-tunées avec LoRA sur notre corpus parallèle français-éwé.  
Cette interface rend le projet plus concret et facilite la démonstration du système lors de la présentation finale.